**Retail Store Sales Dataset**

Phase 1 - Data Loading and Inspection

In [ ]:
# Phase 1 - Data Loading and Inspection

import numpy as np
import pandas as pd

data = pd.read_csv("retail_store_sales.csv")
print(data.head())

  Transaction ID Customer ID       Category          Item  Price Per Unit  \
0    TXN_6867343     CUST_09     Patisserie   Item_10_PAT            18.5   
1    TXN_3731986     CUST_22  Milk Products  Item_17_MILK            29.0   
2    TXN_9303719     CUST_02       Butchers   Item_12_BUT            21.5   
3    TXN_9458126     CUST_06      Beverages   Item_16_BEV            27.5   
4    TXN_4575373     CUST_05           Food   Item_6_FOOD            12.5   

   Quantity  Total Spent  Payment Method Location Transaction Date  \
0      10.0        185.0  Digital Wallet   Online       2024-04-08   
1       9.0        261.0  Digital Wallet   Online       2023-07-23   
2       2.0         43.0     Credit Card   Online       2022-10-05   
3       9.0        247.5     Credit Card   Online       2022-05-07   
4       7.0         87.5  Digital Wallet   Online       2022-10-02   

  Discount Applied  
0             True  
1             True  
2            False  
3              NaN  
4          

In [ ]:
print(data.describe())

       Price Per Unit      Quantity   Total Spent
count    11966.000000  11971.000000  11971.000000
mean        23.365912      5.536380    129.652577
std         10.743519      2.857883     94.750697
min          5.000000      1.000000      5.000000
25%         14.000000      3.000000     51.000000
50%         23.000000      6.000000    108.500000
75%         33.500000      8.000000    192.000000
max         41.000000     10.000000    410.000000


In [ ]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB
None


Phase 2 - Missing Data Handling

In [ ]:
# Phase 2 - Missing Data Handling
# column-wise missing %
# row-wise missing

# print(data.isna().sum(), "\n")
print("column-wise missing % :\n", data.isna().sum()/data.count().max()*100, "\n")
# print(data.isna().sum(axis=1)[:10])
print("row-wise missing % :\n", (data.isna().sum(axis=1)/len(data.columns)*100)[:10])

column-wise missing % :
 Transaction ID       0.000000
Customer ID          0.000000
Category             0.000000
Item                 9.646123
Price Per Unit       4.842942
Quantity             4.803181
Total Spent          4.803181
Payment Method       0.000000
Location             0.000000
Transaction Date     0.000000
Discount Applied    33.391650
dtype: float64 

row-wise missing % :
 0     0.000000
1     0.000000
2     0.000000
3     9.090909
4     0.000000
5    27.272727
6     0.000000
7    27.272727
8     0.000000
9     0.000000
dtype: float64


In [ ]:
# fill numeric → mean/median
# fill categorical → mode

# filling numeric columns with mean/median
print(data[["Price Per Unit","Quantity","Total Spent"]][:5])
cols_to_check = ["Price Per Unit","Quantity","Total Spent"]
rows_idx = data[data[cols_to_check].isna().sum(axis=1)==1].index  # stores index of rows with any one out of the 3 columns missing
print(rows_idx)

# deducing missing values using formula : Total Spent = Quantity * Price Per Unit
def f1(x):
  total = x["Total Spent"]
  quantity = x["Quantity"]
  price_per_unit = x["Price Per Unit"]
  if pd.notna(total) and pd.notna(quantity):
    x["Price Per Unit"] = total /quantity
  elif pd.notna(price_per_unit) and pd.notna(quantity):
    x["Total Spent"] = quantity * price_per_unit
  elif pd.notna(price_per_unit) and pd.notna(total):
    x["Quantity"] = total / price_per_unit

  return x

# def f1(x):
#   # 0th index is Price Per Unit , 1st index is Quantity and 2nd index value is Total Spent
#   if x[2] and x[1]:
#     x[0] = x[2] / x[1]
#   elif x[0] and x[1]:
#     x[2] = x[1] * x[0]
#   else:
#     x[1] = x[2] / x[0]

data.loc[rows_idx, cols_to_check] = data.loc[rows_idx, cols_to_check].apply(f1, axis=1)
rows_idx_updated = data[data[cols_to_check].isna().sum(axis=1)==1].index  # stores index of rows with any one out of the 3 columns missing
print(rows_idx_updated)

   Price Per Unit  Quantity  Total Spent
0            18.5      10.0        185.0
1            29.0       9.0        261.0
2            21.5       2.0         43.0
3            27.5       9.0        247.5
4            12.5       7.0         87.5
Index([    5,    11,    17,    21,    32,   127,   159,   216,   247,   271,
       ...
       12337, 12339, 12365, 12386, 12387, 12435, 12457, 12477, 12491, 12511],
      dtype='int64', length=609)
Index([], dtype='int64')


In [ ]:
# checking count of null values again
print(data.isna().sum())

Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit         0
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64


In [ ]:
# filling nan values in quantity and total spent columns with their mean and median values, respectively
data["Quantity"].fillna(data["Quantity"].mean(), inplace=True)
data["Total Spent"].fillna(data["Total Spent"].median(), inplace=True)

print(data.isna().sum())

Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit         0
Quantity               0
Total Spent            0
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64


/tmp/ipykernel_802/190864932.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data["Quantity"].fillna(data["Quantity"].mean(), inplace=True)
/tmp/ipykernel_802/190864932.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=Tr

In [ ]:
print(data["Item"][:5])

0     Item_10_PAT
1    Item_17_MILK
2     Item_12_BUT
3     Item_16_BEV
4     Item_6_FOOD
Name: Item, dtype: object


In [ ]:
# imputing missing values in item from price_per_unit values wherever possible
unique_items = data["Item"].unique()
# print("unique items:", unique_items)

item_prices = data.drop_duplicates("Item").set_index("Item")["Price Per Unit"].to_dict()
print(item_prices)

item_prices = {value:item for item,value in item_prices.items()}
print(item_prices)
# data["Item"][data["Item"].isna()] = data["Price Per Unit"].map(item_prices)
mask = data["Item"].isna()
data.loc[mask, "Item"] = data.loc[mask, "Price Per Unit"].map(item_prices)
print(data.isna().sum())

{'Item_10_PAT': 18.5, 'Item_17_MILK': 29.0, 'Item_12_BUT': 21.5, 'Item_16_BEV': 27.5, 'Item_6_FOOD': 12.5, nan: 20.0, 'Item_1_FOOD': 5.0, 'Item_16_FUR': 27.5, 'Item_22_BUT': 36.5, 'Item_3_BUT': 8.0, 'Item_2_FOOD': 6.5, 'Item_24_PAT': 39.5, 'Item_16_MILK': 27.5, 'Item_17_PAT': 29.0, 'Item_13_EHE': 23.0, 'Item_7_BEV': 14.0, 'Item_4_EHE': 9.5, 'Item_10_FOOD': 18.5, 'Item_14_FUR': 24.5, 'Item_20_BUT': 33.5, 'Item_25_FUR': 41.0, 'Item_14_FOOD': 24.5, 'Item_22_PAT': 36.5, 'Item_11_FOOD': 20.0, 'Item_6_PAT': 12.5, 'Item_21_EHE': 35.0, 'Item_25_BEV': 41.0, 'Item_23_FOOD': 38.0, 'Item_10_FUR': 18.5, 'Item_11_BEV': 20.0, 'Item_23_BUT': 38.0, 'Item_22_BEV': 36.5, 'Item_10_EHE': 18.5, 'Item_24_BUT': 39.5, 'Item_8_BEV': 15.5, 'Item_3_FOOD': 8.0, 'Item_12_FOOD': 21.5, 'Item_16_CEA': 27.5, 'Item_11_PAT': 20.0, 'Item_16_BUT': 27.5, 'Item_5_CEA': 11.0, 'Item_19_MILK': 32.0, 'Item_23_FUR': 38.0, 'Item_7_FUR': 14.0, 'Item_15_CEA': 26.0, 'Item_6_MILK': 12.5, 'Item_24_CEA': 39.5, 'Item_22_CEA': 36.5, 'Item

In [ ]:
print(data["Discount Applied"][:5])
print(data["Discount Applied"].mode())
data["Discount Applied"].fillna(data["Discount Applied"].mode()[0], inplace=True)

print(data.isna().sum())

0     True
1     True
2    False
3      NaN
4    False
Name: Discount Applied, dtype: object
0    True
Name: Discount Applied, dtype: object
Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
dtype: int64


/tmp/ipykernel_802/651173192.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data["Discount Applied"].fillna(data["Discount Applied"].mode()[0], inplace=True)
/tmp/ipykernel_802/651173192.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data["Discount Applied"].fillna(data["Discount Applied"].mode()[0], 

Phase 3 - Data Cleaning

In [ ]:
print(data.duplicated().sum())

0


In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
print(data.head())

  Transaction ID Customer ID       Category          Item  Price Per Unit  \
0    TXN_6867343     CUST_09     Patisserie   Item_10_PAT            18.5   
1    TXN_3731986     CUST_22  Milk Products  Item_17_MILK            29.0   
2    TXN_9303719     CUST_02       Butchers   Item_12_BUT            21.5   
3    TXN_9458126     CUST_06      Beverages   Item_16_BEV            27.5   
4    TXN_4575373     CUST_05           Food   Item_6_FOOD            12.5   

   Quantity  Total Spent  Payment Method Location Transaction Date  \
0      10.0        185.0  Digital Wallet   Online       2024-04-08   
1       9.0        261.0  Digital Wallet   Online       2023-07-23   
2       2.0         43.0     Credit Card   Online       2022-10-05   
3       9.0        247.5     Credit Card   Online       2022-05-07   
4       7.0         87.5  Digital Wallet   Online       2022-10-02   

   Discount Applied  
0              True  
1              True  
2             False  
3              True  
4     

In [ ]:
# standardizing formats of non-numeric and non-bool columns (strip space, lowercase)
# def str_format(x):
#   tid = x["Transaction ID"]
#   cust_id = x["Customer ID"]
#   category = x["Category"]
#   item = x["Item"]
#   p_method = x["Payment Method"]
#   location = x["Location"]
#   tdate = x["Transaction Date"]

#   for col in [tid, cust_id, category, item, p_method, location, tdate]

non_numeric_cols = list(set(data.columns) - set(cols_to_check))
non_numeric_bool_cols = non_numeric_cols.copy()
non_numeric_bool_cols.remove("Discount Applied")
print(non_numeric_cols)
print(non_numeric_bool_cols)

def str_format_col(col):
  col = col.str.lower()
  col = col.str.strip()
  return col

# data.loc[:,non_numeric_bool_cols] = data.loc[:,non_numeric_bool_cols].str.lower().strip()
data.loc[:,non_numeric_bool_cols] = data.loc[:,non_numeric_bool_cols].apply(str_format_col)
print(data.head())

['Transaction ID', 'Transaction Date', 'Discount Applied', 'Item', 'Category', 'Customer ID', 'Location', 'Payment Method']
['Transaction ID', 'Transaction Date', 'Item', 'Category', 'Customer ID', 'Location', 'Payment Method']
  Transaction ID Customer ID       Category          Item  Price Per Unit  \
0    txn_6867343     cust_09     patisserie   item_10_pat            18.5   
1    txn_3731986     cust_22  milk products  item_17_milk            29.0   
2    txn_9303719     cust_02       butchers   item_12_but            21.5   
3    txn_9458126     cust_06      beverages   item_16_bev            27.5   
4    txn_4575373     cust_05           food   item_6_food            12.5   

   Quantity  Total Spent  Payment Method Location Transaction Date  \
0      10.0        185.0  digital wallet   online       2024-04-08   
1       9.0        261.0  digital wallet   online       2023-07-23   
2       2.0         43.0     credit card   online       2022-10-05   
3       9.0        247.5     

Phase 4 - Feature Engineering

In [ ]:
# ranking the customers with total spent

data["rank_amt"] = data["Total Spent"].rank(method="dense", ascending=False)
print(data.head())

  Transaction ID Customer ID       Category          Item  Price Per Unit  \
0    txn_6867343     cust_09     patisserie   item_10_pat            18.5   
1    txn_3731986     cust_22  milk products  item_17_milk            29.0   
2    txn_9303719     cust_02       butchers   item_12_but            21.5   
3    txn_9458126     cust_06      beverages   item_16_bev            27.5   
4    txn_4575373     cust_05           food   item_6_food            12.5   

   Quantity  Total Spent  Payment Method Location Transaction Date  \
0      10.0        185.0  digital wallet   online       2024-04-08   
1       9.0        261.0  digital wallet   online       2023-07-23   
2       2.0         43.0     credit card   online       2022-10-05   
3       9.0        247.5     credit card   online       2022-05-07   
4       7.0         87.5  digital wallet   online       2022-10-02   

   Discount Applied  rank_amt  
0              True      61.0  
1              True      28.0  
2             False 

In [ ]:
# customers with top ranks (in order of total amount spent)
data_sort = data.copy()
data_sort = data_sort.sort_values(by="rank_amt")
# print(data_sort.head())

print("Top 10 customers with high total amount values:\n", data_sort["Customer ID"].iloc[0:11])

Top 10 customers with high total amount values:
 2153    cust_06
6785    cust_14
9620    cust_14
133     cust_25
1950    cust_01
1505    cust_12
8845    cust_23
1468    cust_17
7272    cust_19
3778    cust_16
8611    cust_08
Name: Customer ID, dtype: object


In [ ]:
print("Top 10 transactions with high total amount values:\n", data_sort.iloc[0:11])

Top 10 transactions with high total amount values:
      Transaction ID Customer ID                       Category          Item  \
2153    txn_5338814     cust_06                  milk products  item_25_milk   
6785    txn_6513125     cust_14                      beverages   item_25_bev   
9620    txn_2386101     cust_14                      furniture   item_25_fur   
133     txn_2953434     cust_25                      furniture   item_25_fur   
1950    txn_1860120     cust_01                      furniture   item_25_fur   
1505    txn_8520397     cust_12  electric household essentials   item_25_ehe   
8845    txn_4208189     cust_23                       butchers   item_25_but   
1468    txn_1938135     cust_17                       butchers   item_25_but   
7272    txn_4858882     cust_19  electric household essentials   item_25_ehe   
3778    txn_7898457     cust_16                      beverages   item_25_bev   
8611    txn_3342443     cust_08                       butchers   ite

In [ ]:
# print("categories within top transactions:", data_sort.loc[data_sort["rank_amt"].astype(int).isin([1,2,3,4,5])])
print("Categories within top transactions:\n", data_sort.loc[data_sort["rank_amt"].astype(int).isin([1,2,3,4,5]), "Category"].unique())

Categories within top transactions:
 ['milk products' 'beverages' 'furniture' 'electric household essentials'
 'butchers' 'food' 'patisserie' 'computers and electric accessories']


In [ ]:
print("Items within top transactions:\n", data_sort.loc[data_sort["rank_amt"].astype(int).isin([1,2,3,4,5]), "Item"].unique())

Items within top transactions:
 ['item_25_milk' 'item_25_bev' 'item_25_fur' 'item_25_ehe' 'item_25_but'
 'item_25_food' 'item_25_pat' 'item_25_cea' 'item_24_fur' 'item_24_food'
 'item_24_ehe' 'item_24_bev' 'item_24_pat' 'item_24_milk' 'item_24_but'
 'item_23_ehe' 'item_23_cea' 'item_23_pat' 'item_23_but' 'item_23_fur'
 'item_23_milk' 'item_23_food' 'item_23_bev' 'item_22_food' 'item_22_cea'
 'item_22_pat' 'item_22_but' 'item_22_bev' 'item_22_milk' 'item_22_ehe'
 'item_22_fur']


Phase 5 - Outlier Detection

In [ ]:
# detecting outliers on the basis of price per unit using IQR method

q1 = data["Price Per Unit"].quantile(0.25)
q3 = data["Price Per Unit"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - (1.5*iqr)
upper_bound = q3 + (1.5*iqr)
# outliers = data[((data["Price Per Unit"])<(q1-(1.5*iqr))) | ((data["Price Per Unit"])>(q3+(1.5*iqr)))]
outliers = data[(data["Price Per Unit"]<lower_bound) | (data["Price Per Unit"]>upper_bound)]
print(outliers[:5])

Empty DataFrame
Columns: [Transaction ID, Customer ID, Category, Item, Price Per Unit, Quantity, Total Spent, Payment Method, Location, Transaction Date, Discount Applied, rank_amt]
Index: []


In [ ]:
# detecting outliers on the basis of total amount spent using IQR method
q1 = data["Total Spent"].quantile(0.25)
q3 = data["Total Spent"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - (1.5*iqr)
upper_bound = q3 + (1.5*iqr)
# outliers = data[((data["Price Per Unit"])<(q1-(1.5*iqr))) | ((data["Price Per Unit"])>(q3+(1.5*iqr)))]
outliers_amt = data[(data["Total Spent"]<lower_bound) | (data["Total Spent"]>upper_bound)]
print(outliers_amt[:5])
# print(len(outliers_amt), outliers_amt.shape)
print(outliers_amt.shape)

    Transaction ID Customer ID       Category          Item  Price Per Unit  \
27     txn_1599706     cust_14      furniture   item_25_fur            41.0   
120    txn_8215469     cust_17  milk products  item_23_milk            38.0   
129    txn_1112365     cust_19           food  item_24_food            39.5   
133    txn_2953434     cust_25      furniture   item_25_fur            41.0   
135    txn_1249742     cust_05  milk products  item_24_milk            39.5   

     Quantity  Total Spent  Payment Method  Location Transaction Date  \
27       10.0        410.0     credit card  in-store       2024-03-24   
120      10.0        380.0  digital wallet  in-store       2024-03-16   
129      10.0        395.0            cash    online       2023-09-24   
133      10.0        410.0     credit card  in-store       2023-08-10   
135      10.0        395.0  digital wallet    online       2024-03-21   

     Discount Applied  rank_amt  
27               True       1.0  
120              T

Phase 6 - Encoding

In [ ]:
# encoding categories of products purchased (one-hot encoding)
data_encoding = data.copy()
print(data_encoding.head())
category_dummies = pd.get_dummies(data["Category"], prefix="Category_", drop_first=True)
print(category_dummies[:5])

data_encoding  = data_encoding.join(category_dummies)
print(data_encoding.head())

  Transaction ID Customer ID       Category          Item  Price Per Unit  \
0    txn_6867343     cust_09     patisserie   item_10_pat            18.5   
1    txn_3731986     cust_22  milk products  item_17_milk            29.0   
2    txn_9303719     cust_02       butchers   item_12_but            21.5   
3    txn_9458126     cust_06      beverages   item_16_bev            27.5   
4    txn_4575373     cust_05           food   item_6_food            12.5   

   Quantity  Total Spent  Payment Method Location Transaction Date  \
0      10.0        185.0  digital wallet   online       2024-04-08   
1       9.0        261.0  digital wallet   online       2023-07-23   
2       2.0         43.0     credit card   online       2022-10-05   
3       9.0        247.5     credit card   online       2022-05-07   
4       7.0         87.5  digital wallet   online       2022-10-02   

   Discount Applied  rank_amt  
0              True      61.0  
1              True      28.0  
2             False 

Phase 7 - Final Cleaned Dataset

In [ ]:
print(data.isna().sum())

Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
rank_amt            0
dtype: int64


In [ ]:
print(data.describe())

       Price Per Unit      Quantity   Total Spent      rank_amt
count    12575.000000  12575.000000  12575.000000  12575.000000
mean        23.369304      5.536380    128.636581    115.147515
std         10.748728      2.788398     92.557580     63.285456
min          5.000000      1.000000      5.000000      1.000000
25%         14.000000      3.000000     55.000000     62.000000
50%         23.000000      5.536380    108.500000    116.000000
75%         33.500000      8.000000    184.000000    169.000000
max         41.000000     10.000000    410.000000    227.000000


In [ ]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              12575 non-null  object 
 4   Price Per Unit    12575 non-null  float64
 5   Quantity          12575 non-null  float64
 6   Total Spent       12575 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  12575 non-null  bool   
 11  rank_amt          12575 non-null  float64
dtypes: bool(1), float64(4), object(7)
memory usage: 1.1+ MB
None


In [ ]:
data.to_csv("cleaned_retail_store_sales.csv")